<a href="https://colab.research.google.com/github/BerkeleyExpertSystemTechnologiesLab/Squishy-Methane-Analysis/blob/jberry/Squish_Robot_Quant_Model_v5_1_mm_%2B_image_transforms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Description

This model was produced by and for Squishy Robotics for the task of identifying and classifying methane leaks.


This model was made in conjunction with a synthetic dataset of 2 channel, 240 by 320 greyscale images of methane leaks
(2 x 240 x 320)
The first channel is a greyscale background image and the second channel is a greyscale gas plume image.



This model is experimental and uses the Optuna Hyperparameter Optimizer to search for successful hyperparameters (Learning Rate, Optimizer, Batch Size, Dropout %, etc...) and different optimizers. As such if you want to test a specific Model architecture you need to comment out the Optuna code and run a train/test on that specific model.

In [1]:
pip install optuna #Hyperparameter Optimizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 12.9 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np

from collections import defaultdict
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from torch.utils.data import random_split

# Hyperparameter Search
import optuna

import json
import glob


In [ ]:
torch.__version__

In [ ]:
torchvision.__version__

In [ ]:
CHANNELS = 2
NUM_HEADS = 8
DEPTH = 6

In [3]:
# This may take several minutes, the synthetic dataset can be large
!unzip -q Final_Dataset.zip

## Print out the shape of the data

In [4]:
# Loading an example file to demonstrate the dimensions
# This file might not exist, change the name to one that does to show the
# dimensions
file_path = './Final_Dataset/data/class_0/1237_frame_1004_class_0.npy'
sample_data = np.load(file_path)
print(f"Shape of preprocessed sample data: {sample_data.shape}")
print(f"Data type of preprocessed sample data: {sample_data.dtype}")

# GasVid synthetic processed dataset should be 2 channels, 240x320 in dimension

FileNotFoundError: [Errno 2] No such file or directory: './Final_Dataset/data/class_0/1237_frame_1004_class_0.npy'

In [5]:
# Assuming the data is in 'Final_Dataset/data' and class folders are named 'class_0' ... 'class_7'
data_dir = 'Final_Dataset/data'
classes = sorted(os.listdir(data_dir))
print(f"Classes: {classes}")

Classes: ['class_0', 'class_1', 'class_2', 'class_3', 'class_4', 'class_5', 'class_6', 'class_7']


## Create a dataset and dataloader

In [6]:
class Multi_Modal_Dataset(Dataset):
    def __init__(self, numpy_files, json_files, labels, transform=None):
        """
        numpy_dir points to all the numpy 2 channel frames that were collected
          from METEC. This is designed to be 1st Channel Greyscale image of
          background, 2nd channel is just the gas plume scaled to some ppm
        json_dir points to all the metadata (ppm, distance, etc) that was
          collected from METEC or estimated using BEST Labs algorithms
        """
        self.numpy_files = numpy_files
        self.json_files = json_files
        self.labels = labels
        self.transform = transform


    def __len__(self):
      return len(self.numpy_files)


    def __getitem__(self, idx):
      numpy_path = self.numpy_files[idx]
      image_data = np.load(numpy_path)
      image_tensor = torch.from_numpy(image_data).float()

      if self.transform:
        image_tensor = self.transform(image_tensor)

      json_path = self.json_files[idx]
      with open(json_path, 'r') as f:
        metadata = json.load(f)

      metadat_features = self._extract_metadata_features(metadata)
      metadata_tensor = torch.tensor(metadat_features, dtype=torch.float32)

      label = self.labels[idx]

      return image_tensor, metadata_tensor, label


    def _extract_metadata_features(self, metadata):
      """
      Extracts a few entries from the metadata.
        For now:
          distance
          ppm
        In the future
          windspeed
          angle?
      """

      features = []

      # If the features exist, extract them, else place 0.0
      # Print warning statements if unable to retrieve the data
      distance = metadata.get("distance_m", None)
      if distance is None or distance == 0.0:
          print(f"WARNING: Invalid or missing distance_m value: {distance}")
          print(f"  Metadata keys available: {list(metadata.keys())}")
          features.append(0.0)
      else:
          features.append(distance)

      ppm = metadata.get("ppm", None)
      if ppm is None:
          print(f"WARNING: Missing ppm value")
          print(f"  Metadata keys available: {list(metadata.keys())}")
          features.append(0.0)
      else:
          features.append(ppm)

      return features

In [7]:
numpy_dir = "./Final_Dataset/data"
json_dir = "./Final_Dataset/metadata"

all_numpy_files = []
all_json_files = []
all_labels = []

print(f"Looking in: {numpy_dir}")
print(f"Directory exists: {os.path.exists(numpy_dir)}\n")

# Load each class separatley, collect the numpy and json files for a certain
# class at the same time
for class_idx in range(8):
    numpy_class_dir = os.path.join(numpy_dir, f"class_{class_idx}")
    json_class_dir = os.path.join(json_dir, f"class_{class_idx}")

    numpy_files_in_class = sorted(glob.glob(os.path.join(numpy_class_dir, "*.npy")))

    print(f"Class {class_idx}: Found {len(numpy_files_in_class)} files")

    for numpy_file in numpy_files_in_class:
        base_name = os.path.splitext(os.path.basename(numpy_file))[0]
        video_id = base_name.split('_')[0]


        json_filename = f"{video_id}_class_{class_idx}.json"
        json_file = os.path.join(json_class_dir, json_filename)

        if os.path.exists(json_file):
            all_numpy_files.append(numpy_file)
            all_json_files.append(json_file)
            all_labels.append(class_idx)
        else:
            print(f"WARNING: JSON missing for {base_name}")

print(f"\n{'='*60}")
print(f"TOTAL: {len(all_numpy_files)} numpy files")
print(f"TOTAL: {len(all_json_files)} json files")
print(f"{'='*60}\n")

# Only continue if we have files
if len(all_numpy_files) == 0:
    raise ValueError("!!!No files found!!! Check your paths above.")

# Now continue with video splitting
video_to_indices = defaultdict(list)
for idx, numpy_file in enumerate(all_numpy_files):
    video_id = os.path.basename(numpy_file).split('_')[0]
    video_to_indices[video_id].append(idx)

print(f"Number of unique videos: {len(video_to_indices)}")
print(f"Video IDs: {sorted(video_to_indices.keys())}\n")


Looking in: ./Final_Dataset/data
Directory exists: True

Class 0: Found 5420 files
Class 1: Found 5407 files
Class 2: Found 5371 files
Class 3: Found 5376 files
Class 4: Found 5391 files
Class 5: Found 5391 files
Class 6: Found 5415 files
Class 7: Found 5404 files

TOTAL: 43175 numpy files
TOTAL: 43175 json files

Number of unique videos: 28
Video IDs: ['1237', '1238', '1239', '1240', '1241', '1242', '1467', '1468', '1469', '1470', '1471', '1472', '2559', '2560', '2561', '2562', '2563', '2564', '2566', '2567', '2568', '2569', '2571', '2578', '2579', '2580', '2581', '2583']



In [8]:
video_to_indices = defaultdict(list) #Make an empty dictionary of lists

for idx, numpy_file in enumerate(all_numpy_files):
    video_id = os.path.basename(numpy_file).split('_')[0] #Extract 4 digit code from numpy filename
    video_to_indices[video_id].append(idx)

video_ids = list(video_to_indices.keys())

# Split the video into train and test
train_vids, test_vids = train_test_split(video_ids, test_size=0.2, random_state=42)

# Verify no overlap
overlap = set(train_vids) & set(test_vids)
if overlap:
    print(f"\nVideos overlap: {overlap}")
else:
    print(f"\nNo video overlap - train and test are separate")

train_indices = []
test_indices = []

for vid in train_vids:
    train_indices.extend(video_to_indices[vid])
for vid in test_vids:
    test_indices.extend(video_to_indices[vid])

# Create file lists
train_numpy = [all_numpy_files[i] for i in train_indices]
train_json = [all_json_files[i] for i in train_indices]
train_labels_list = [all_labels[i] for i in train_indices]

test_numpy = [all_numpy_files[i] for i in test_indices]
test_json = [all_json_files[i] for i in test_indices]
test_labels_list = [all_labels[i] for i in test_indices]



No video overlap - train and test are separate


### Image Transformations

In [9]:
# Augmentation section
# https://docs.pytorch.org/vision/0.13/transforms.html
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1),
    ),
    transforms.RandomApply([
        transforms.GaussianBlur(
            kernel_size=3,
            sigma=(0.1, 2.0)
        )
    ], p=0.3)
])

#During testing don't use augmentation
test_transforms = None

In [10]:
# SHOW FINAL SPLIT STATISTICS
print(f"\n{'='*60}")
print("DATASET STATISTICS")
print("="*90)

print(f"\nTRAINING SET:")
print(f"   Total samples: {len(train_numpy)}")
print(f"   From {len(train_vids)} videos: {sorted(train_vids)}")

# Count samples per class in training
train_class_counts = Counter(train_labels_list)
print(f"\n   Samples per class:")
for class_id in range(8):
    count = train_class_counts.get(class_id, 0)
    percentage = (count / len(train_numpy) * 100) if len(train_numpy) > 0 else 0
    print(f"      Class {class_id}: {count:5d} samples ({percentage:5.2f}%)")

print(f"\nTEST SET:")
print(f"   Total samples: {len(test_numpy)}")
print(f"   From {len(test_vids)} videos: {sorted(test_vids)}")

# Count samples per class in testing
test_class_counts = Counter(test_labels_list)
print(f"\n   Samples per class:")
for class_id in range(8):
    count = test_class_counts.get(class_id, 0)
    percentage = (count / len(test_numpy) * 100) if len(test_numpy) > 0 else 0
    print(f"      Class {class_id}: {count:5d} samples ({percentage:5.2f}%)")

# VERIFY ALL CLASSES PRESENT
print(f"\n{'='*70}")
print("VERIFICATION")
print("="*70)

train_classes = set(train_labels_list)
test_classes = set(test_labels_list)
missing_train = set(range(8)) - train_classes
missing_test = set(range(8)) - test_classes

if missing_train:
    print(f"WARNING: Training missing classes {missing_train}")
else:
    print(f"Training set has all 8 classes")

if missing_test:
    print(f"WARNING: Testing missing classes {missing_test}")
else:
    print(f"Test set has all 8 classes")

# Show train/test split ratio
total_samples = len(train_numpy) + len(test_numpy)
train_ratio = len(train_numpy) / total_samples * 100
test_ratio = len(test_numpy) / total_samples * 100
print(f"\nSplit ratio: {train_ratio:.1f}% train / {test_ratio:.1f}% test")

print(f"\n{'='*70}")
print("DATA SPLIT COMPLETE AND VERIFIED")
print("="*70)



DATASET STATISTICS

TRAINING SET:
   Total samples: 33893
   From 22 videos: ['1238', '1239', '1240', '1241', '1242', '1467', '1468', '1471', '1472', '2560', '2561', '2562', '2563', '2564', '2566', '2567', '2568', '2571', '2578', '2579', '2581', '2583']

   Samples per class:
      Class 0:  4251 samples (12.54%)
      Class 1:  4243 samples (12.52%)
      Class 2:  4217 samples (12.44%)
      Class 3:  4208 samples (12.42%)
      Class 4:  4233 samples (12.49%)
      Class 5:  4237 samples (12.50%)
      Class 6:  4255 samples (12.55%)
      Class 7:  4249 samples (12.54%)

TEST SET:
   Total samples: 9282
   From 6 videos: ['1237', '1469', '1470', '2559', '2569', '2580']

   Samples per class:
      Class 0:  1169 samples (12.59%)
      Class 1:  1164 samples (12.54%)
      Class 2:  1154 samples (12.43%)
      Class 3:  1168 samples (12.58%)
      Class 4:  1158 samples (12.48%)
      Class 5:  1154 samples (12.43%)
      Class 6:  1160 samples (12.50%)
      Class 7:  1155 samples

In [11]:
train_dataset = Multi_Modal_Dataset(train_numpy,
                                    train_json,
                                    train_labels_list,
                                    transform=train_transforms)
test_dataset = Multi_Modal_Dataset(test_numpy,
                                   test_json,
                                   test_labels_list,
                                   transform=test_transforms)

# Define CvT model

In [ ]:
# With image size of 2 channel 240x320 and patch size of 16
# there will be 300 patches (240 / 16) * (320 / 16) = 300 (one patch for combined channels)
class PatchEmbedding(nn.Module):
    def __init__(self, img_height = 240, img_width = 320, patch_size=16, in_channels=2, embed_dim=128):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.proj(x).flatten(2).transpose(1, 2)
        return x


# +1 to seq_len for [CLS] token
# nn.parameter makes pos_embed learnable during training
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, seq_len):
        super().__init__()
        
        self.pos_embed = nn.Parameter(torch.randn(1, seq_len + 1, embed_dim))  

    def forward(self, x):
        return x + self.pos_embed

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads)

    def forward(self, x):
        return self.attn(x, x, x)[0]


class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout = 0.1):
        super().__init__()
        self.attn = MultiHeadAttention(embed_dim, num_heads)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim)
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        x = x + self.dropout(self.mlp(self.norm2(x)))
        return x


# By default the transformer models output their predictions to a classifier, the head
# however since this model is multi-modal we are combining the outputs from this transformer
# with the metadata, and then making a third smaller network and classifying that output
# the include head section below is a hack around having the ViT output a prediction 
# directly to a classifier. 
class VisionTransformer(nn.Module):
    def __init__(self, img_height = 240, img_width = 320, patch_size=16, 
                 num_classes=8, embed_dim=768, num_heads=8, depth=6, 
                 mlp_dim=1024, in_channels = 2, dropout = 0.3, include_head = True):
        super().__init__()
        self.patch_embedding = PatchEmbedding(img_height, img_width, patch_size, in_channels, embed_dim)

        seq_len = (img_height // patch_size) * (img_width // patch_size) 
        self.pos_encoding = PositionalEncoding(embed_dim, seq_len)

        self.transformer_blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_dim, dropout) for _ in range(depth)
        ])
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        self.include_head = include_head
        if include_head:
            self.mlp_head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.size(0)
        x = self.patch_embedding(x)             # (B, 300, embed_dim)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)   # (B, 301, embed_dim) 
        x = self.pos_encoding(x)

        for block in self.transformer_blocks:
            x = block(x)
        cls_output = x[:, 0]

        if self.include_head:
            return self.mlp_head(cls_output)  # (B, num_classes), output classifier decision directly
        else:
            return cls_output # (B, embed_dim), output the network without classifier

class ConvolutionalTokenEmbedding(nn.Module):

    def __init__(self, in_channels, embed_dim, kernel_size = 7, stride= 4, padding = 2):
        super().__init__()
        self.proj = nn.Conv2d( in_channels, embed_dim, kernel_size = kernel_size, stride = stride, padding = padding)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2) # converts (B, H, W, C) to (B, HxW, C)
        x = self.norm(x)
        return x, (H,W)

class ConvolutionalProjection(nn.Module):
    def __init__(self, embed_dim, kernel_size = 3, stride= 1, padding = 1, groups = None):
        super().__init__()

        if groups is None:
            groups = embed_dim

        self.depthwise_conv = nn.Conv2d(
            embed_dim,
            embed_dim,
            kernel_size = kernel_size,
            stride = stride,
            padding = padding,
            groups = groups
        )

        self.pointwise_conv = nn.Conv2d( embed_dim, embed_dim, kernel_size = 1)
        self.norm = nn.LayerNorm( embed_dim )

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.transpose(1,2).reshape(B, C, H, W)
        x = self.depthwise_conv(x)
        x = self.pointwise_conv(x)
        x = x.flatten(2).transpose( 1, 2 ) # converts (B, H, W, C) to (B, HxW, C)
        x = self.norm(x)
        return x

class ConvolutionalMultiHeadAttention(nn.Module):
    def __init__( self, embed_dim, num_heads, kernel_size = 3, stride= 1, padding = 1, dropout = 0.1):
        super().__init__()
        assert embed_dim % num_heads == 0 #embed_dim mod numb_heads must equal zero

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_proj = ConvolutionalProjection( embed_dim, kernel_size, stride, padding)
        self.k_proj = ConvolutionalProjection( embed_dim, kernel_size, stride, padding)
        self.v_proj = ConvolutionalProjection( embed_dim, kernel_size, stride, padding)

        self.out_proj = nn.Linear( embed_dim, embed_dim )
        self.dropout = nn.Dropout(dropout)

    def forward (self, x, H, W):
        B, N, C = x.shape

        Q = self.q_proj(x, H, W)
        K = self.k_proj(x, H, W)
        V = self.v_proj(x, H, W)

        Q = Q.reshape( B, N, self.num_heads, self.head_dim).transpose(1,2)
        K = K.reshape( B, N, self.num_heads, self.head_dim).transpose(1,2)
        V = V.reshape( B, N, self.num_heads, self.head_dim).transpose(1,2)

        scale = self.head_dim ** -0.5
        attn = ( Q @ K.transpose(-2, -1)) * scale
        attn = attn.softmax(dim = -1)
        attn = self.dropout(attn)

        x = (attn @ V).transpose(1, 2).reshape(B, N, C)
        x = self.out_proj(x)

        return x

class ConvolutionalTransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout = 0.1, 
                 kernel_size = 3, stride = 1, padding = 1):
        super().__init__()

        self.attn = ConvolutionalMultiHeadAttention(
            embed_dim, num_heads, kernel_size, stride, padding, dropout
        )

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

        self.norm1 = nn.LayerNorm( embed_dim )
        self.norm2 = nn.LayerNorm( embed_dim )

    def forward(self, x, H, W):
        # Attention Block
        x = x + self.attn(self.norm1(x), H, W)
        # MLP Block
        x = x + self.mlp(self.norm2(x))
        return x

class CvTStage(nn.Module):
    def __init__(self, in_channels, embed_dim, num_heads, depth, mlp_dim,
                 dropout = 0.1, kernel_size = 7, stride = 4, padding = 2,
                 attn_kernel_size = 3, attn_stride = 1, attn_padding = 1, 
                 is_first_stage = False):
        super().__init__()

        self.is_first_stage = is_first_stage

        self.token_embedding = ConvolutionalTokenEmbedding(
            in_channels, embed_dim, kernel_size, stride, padding
        )

        self.transformer_blocks = nn.ModuleList([
            ConvolutionalTransformerBlock(
                embed_dim, num_heads, mlp_dim, dropout,
                attn_kernel_size, attn_stride, attn_padding
            ) for _ in range(depth)
        ])

    def forward(self, x, prev_H=None, prev_W=None):
        if self.is_first_stage:
            # First stage: input is image tensor (B, C, H, W)
            x, (H, W) = self.token_embedding(x)
        else:
            # Later stages: input is token sequence (B, N, C)
            # Need to reshape back to spatial using previous stage's dimensions
            B, N, C = x.shape
            if prev_H is None or prev_W is None:
                # Fallback: try to infer (but this is not ideal)
                H = W = int(N ** 0.5)
                if H * W != N:
                    # If not square, we need to know the actual dimensions
                    raise ValueError(f"Cannot infer spatial dimensions from N={N}. Need prev_H and prev_W.")
            else:
                H, W = prev_H, prev_W
            
            # Reshape tokens back to spatial format
            x = x.transpose(1, 2).reshape(B, C, H, W)
            # Apply new token embedding (which will downsample)
            x, (H, W) = self.token_embedding(x)
        
        # Apply transformer blocks
        for block in self.transformer_blocks:
            x = block(x, H, W)
        
        return x, (H, W)


In [ ]:
class ConvolutionalVisionTransformer(nn.Module):
    def __init__(self, img_height=240, img_width=320, in_channels=2,
                embed_dims=[64, 128, 256], num_heads=[1, 2, 4], 
                depths=[2, 2, 2], mlp_dims=[256, 512, 1024],
                dropout=0.1, num_classes=8, include_head=True):
        super().__init__()
            
        self.embed_dims = embed_dims
        self.include_head = include_head

        self.stage0 = CvTStage(
            in_channels = in_channels,
            embed_dim = embed_dims[0],
            num_heads = num_heads[0],
            depth = depths[0],
            mlp_dim = mlp_dims[0],
            dropout = dropout,
            kernel_size = 7,
            stride = 4,
            padding = 2,
            is_first_stage = True
        )

        self.stage1 = CvTStage(
            in_channels = embed_dims[0],
            embed_dim = embed_dims[1],
            num_heads = num_heads[1],
            depth = depths[1],
            mlp_dim = mlp_dims[1],
            dropout = dropout,
            kernel_size = 3,
            stride = 2,
            padding = 1,
            is_first_stage = False
        )

        self.stage2 = CvTStage(
            in_channels = embed_dims[1],
            embed_dim = embed_dims[2],
            num_heads = num_heads[2],
            depth = depths[2],
            mlp_dim = mlp_dims[2],
            dropout = dropout,
            kernel_size = 3,
            stride = 2,
            padding = 1,
            is_first_stage = False
        )

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dims[2]))

        if include_head:
            self.mlp_head = nn.Linear(embed_dims[2], num_classes)
        else:
            self.mlp_head = None

    def forward(self, x):
        B = x.size(0)
        
        # Stage 0
        x, (H0, W0) = self.stage0(x)  # No prev_H/W needed for first stage
        
        # Stage 1
        x, (H1, W1) = self.stage1(x, prev_H=H0, prev_W=W0)
        
        # Stage 2
        x, (H2, W2) = self.stage2(x, prev_H=H1, prev_W=W1)
        
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        cls_output = x[:, 0]
        
        if self.include_head and self.mlp_head is not None:
            return self.mlp_head(cls_output)
        else:
            return cls_output

In [ ]:
def objective(trial):

    #############################
    # All Hyperparameters Tested
    #############################
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'AdamW'])
    momentum = trial.suggest_float('momentum', 0.0, 0.99) if optimizer_name in ['SGD'] else 0.0
    weight_decay = trial.suggest_float('weight_decay', 0.0, 0.01)
    # hidden_size = trial.suggest_int('hidden_size', 64, 256)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    num_epochs = trial.suggest_int('num_epochs', 5, 10)
    fc_drop_rate = trial.suggest_float('fc_drop_rate', 0.2, 0.6)
    patch_size = trial.suggest_categorical('patch_size', [8, 16, 20])
    embed_dim = trial.suggest_categorical('embed_dim', [128, 256, 384])
    depth = trial.suggest_int('depth', 4, 8)
    num_heads = trial.suggest_categorical('num_heads', [4, 8])
    mlp_dim = trial.suggest_int('mlp_dim', 256, 1024)
    vit_drop_rate = trial.suggest_float('vit_drop_rate', 0.0, 0.3)

    #####################
    # Define the Model
    #####################
    class MultiModeViT(nn.Module):
        def __init__(self, img_height = 240, img_width = 320, patch_size = 16,
                    num_classes = 8, embed_dim = 256, num_heads = 8, depth = 6,
                    mlp_dim = 512, in_channels = 2, num_metadata_feats = 2, fc_drop_rate = 0.3, vit_drop_rate = 0.3):
            super(MultiModeViT, self).__init__()
            
            # CvT for images only (
            # Use embed_dim for final stage embedding dimension
            # Scale other stages proportionally
            embed_dims = [embed_dim // 4, embed_dim // 2, embed_dim]
            num_heads_list = [num_heads // 4, num_heads // 2, num_heads]
            depths = [depth // 3, depth // 3, depth - 2 * (depth // 3)]  # Distribute depth across 3 stages
            mlp_dims = [mlp_dim // 2, mlp_dim, mlp_dim * 2]
            
            # Ensure minimum values
            embed_dims = [max(32, d) for d in embed_dims]
            num_heads_list = [max(1, h) for h in num_heads_list]
            depths = [max(1, d) for d in depths]
            mlp_dims = [max(64, d) for d in mlp_dims]
            
            self.cvt = ConvolutionalVisionTransformer(
                img_height = img_height,
                img_width = img_width,
                in_channels = in_channels,
                embed_dims = embed_dims,
                num_heads = num_heads_list,
                depths = depths,
                mlp_dims = mlp_dims,
                dropout = vit_drop_rate,
                num_classes = embed_dim,  # Output embedding dimension, not num_classes
                include_head = False
            )

            # Smaller Neural Net for metadata only
            self.metadata_fc = nn.Sequential(
                nn.Linear(num_metadata_feats, 64),
                nn.ReLU(),
                nn.Dropout(fc_drop_rate),
                nn.Linear(64, 64)
            )
            
            # Classifier combines metadata NN and image CvT outputs
            self.classifier = nn.Sequential(
                nn.Linear(embed_dim + 64, 128),
                nn.ReLU(),
                nn.Dropout(fc_drop_rate),
                nn.Linear(128, num_classes)
            )

        def forward(self, image, metadata):
            cvt_out = self.cvt(image)  # Changed from self.vit to self.cvt
            meta_out = self.metadata_fc(metadata)
            combined = torch.cat([cvt_out, meta_out], dim = 1)
            output = self.classifier(combined)

            return output



    model = MultiModeViT(
        patch_size = patch_size,
        embed_dim = embed_dim,
        num_heads = num_heads,
        depth = depth,
        mlp_dim = mlp_dim,
        fc_drop_rate = fc_drop_rate,
        vit_drop_rate = vit_drop_rate
    )

    ###############################
    # Define optimizer
    ###############################
    if optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    elif optimizer_name == 'RMSprop':
        optimizer = optim.RMSprop(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    elif optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'Adadelta':
        optimizer = optim.Adadelta(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "Muon":
        optimizer = optim.Muon(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unknown optimizer name: {optimizer_name}")

    criterion = nn.CrossEntropyLoss()

    ##########################################
    # Create DataLoaders with trial batch_size
    ##########################################
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    ###############################
    # Train the model
    ###############################
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)


    print(f"\n{'='*70}")
    print(f"Trial {trial.number} | lr={lr:.6f} | optimizer={optimizer_name} | "
      f"batch={batch_size} | embed_dim={embed_dim} | depth={depth}")
    print(f"{'='*70}")


    model.train()
    train_correct = 0
    train_total = 0
    for epoch in range(num_epochs):
        train_correct = 0
        train_total = 0
        train_loss = 0.0
        num_batches = 0

        for images, metadata, labels in train_loader:
            images = images.to(device)
            metadata = metadata.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images, metadata)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # Calculate loss
            train_loss += loss.item()
            num_batches += 1

            # Calculate training accuracy
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        #Print out training during each epoch
        train_accuracy = train_correct / train_total
        avg_train_loss = train_loss / num_batches

        # Print training accuracy for this epoch
        print(f"Epoch [{epoch+1:2d}/{num_epochs}] Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.4f}")

    #####################
    # Evaluate the model
    #####################
    model.eval()
    correct, total = 0, 0
    val_loss = 0.0
    num_val_batches = 0
    with torch.no_grad():

        for images, metadata, labels in test_loader:
            images = images.to(device)
            metadata = metadata.to(device)
            labels = labels.to(device)

            # Calculate validation loss
            outputs = model(images, metadata)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            num_val_batches += 1

            # Calculate Validation Accuract
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    avg_val_loss = val_loss / num_val_batches

    print(f"Validation Loss: {avg_val_loss:.4f} | Validation Acc: {accuracy:.4f}")
    print(f"{'='*70}\n")

    return accuracy


## Define the Optuna Objective Function


In [ ]:
# Create a study object and specify the direction of optimization (maximize accuracy)
study = optuna.create_study(direction='maximize',
                             pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5))

# Run the optimization
study.optimize(objective, n_trials = 30)

# Print the best hyperparameters found
print("Best hyperparameters: ", study.best_params)

# Print the best accuracy found
print("Best accuracy: ", study.best_value)

# Plot the visualization
optuna.visualization.plot_param_importances(study).show()

# Run more trials
# study.optimize(objective, n_trials=20)

[I 2025-11-15 22:04:21,446] A new study created in memory with name: no-name-387f8978-3b43-4ded-ad10-e378489e6edb



Trial 0 | lr=0.001419 | optimizer=Adam | batch=128 | hidden=93
Epoch [ 1/9] Train Loss: 1.8205 | Train Acc: 0.2642
Epoch [ 2/9] Train Loss: 1.4805 | Train Acc: 0.3890
Epoch [ 3/9] Train Loss: 1.1005 | Train Acc: 0.5404
Epoch [ 4/9] Train Loss: 0.9707 | Train Acc: 0.5894
Epoch [ 5/9] Train Loss: 1.0090 | Train Acc: 0.5784
Epoch [ 6/9] Train Loss: 0.8940 | Train Acc: 0.6202
Epoch [ 7/9] Train Loss: 0.9212 | Train Acc: 0.6193
Epoch [ 8/9] Train Loss: 0.9106 | Train Acc: 0.6159
Epoch [ 9/9] Train Loss: 0.8427 | Train Acc: 0.6441


[I 2025-11-15 23:18:34,596] Trial 0 finished with value: 0.8267614738202973 and parameters: {'lr': 0.001418995956648577, 'optimizer': 'Adam', 'weight_decay': 0.0013126042873484256, 'hidden_size': 93, 'batch_size': 128, 'num_epochs': 9, 'fc_drop_rate': 0.20239350993793923, 'cnn_drop_rate': 0.1373207584756213}. Best is trial 0 with value: 0.8267614738202973.


Validation Loss: 0.6224 | Validation Acc: 0.8268


Trial 1 | lr=0.049805 | optimizer=Adam | batch=64 | hidden=249


# Sources:
### Hyperparameter Tuning with Optuna:
https://medium.com/@taeefnajib/hyperparameter-tuning-using-optuna-c46d7b29a3e
https://optuna.org/#code_examples

### Multi-Modal ML Models
https://www.nature.com/articles/s41598-025-14901-4
https://www.reddit.com/r/MachineLearning/comments/nziumg/combining_images_and_other_numeric_features_in_a/
https://pyimagesearch.com/2019/02/04/keras-multiple-inputs-and-mixed-data/

### ViT Models
https://www.geeksforgeeks.org/deep-learning/building-a-vision-transformer-from-scratch-in-pytorch/
https://www.youtube.com/watch?v=7o1jpvapaT0&t=2924s
